Camada Bronze (Ingestão & Auditoria)

In [0]:
from pyspark.sql import functions as F

CATALOGO = "mvp"
ESQUEMA = "staging"
VOLUME = "diabetes"
NOME_ARQUIVO = "diabetes_risk_prediction_dataset.csv"

CAMINHO_CSV = f"/Volumes/{CATALOGO}/{ESQUEMA}/{VOLUME}/{NOME_ARQUIVO}"
TABELA_BRONZE = f"{CATALOGO}.{ESQUEMA}.bronze_diabetes_raw"

spark.sql(f"USE CATALOG {CATALOGO}")
spark.sql(f"USE SCHEMA {ESQUEMA}")

# Leitura e Adição de Metadados de Auditoria
df_bronze = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(CAMINHO_CSV) \
    .withColumn("_ingestion_datetime", F.current_timestamp()) \
    .withColumn("_source_file", F.lit(CAMINHO_CSV))

# Persistência em Delta Lake
df_bronze.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABELA_BRONZE)

In [0]:
%sql
SELECT * FROM mvp.staging.bronze_diabetes_raw LIMIT 10;

Camada Silver (Limpeza & Padronização)

In [0]:
from pyspark.sql import functions as F

CATALOGO = "mvp"
ESQUEMA = "staging"

TABELA_BRONZE = f"{CATALOGO}.{ESQUEMA}.bronze_diabetes_raw"
TABELA_SILVER = f"{CATALOGO}.{ESQUEMA}.silver_diabetes_clean"

df_bronze = spark.table(TABELA_BRONZE)

# Normalização de Nomes de Colunas
for c in df_bronze.columns:
    df_bronze = df_bronze.withColumnRenamed(c, c.strip().lower())

colunas = df_bronze.columns

def obter_coluna(opcoes):
    for op in opcoes:
        if op in colunas:
            return op
    return None

col_glicose   = obter_coluna(["fastingbloodsugar", "fasting_blood_sugar", "glucose", "glicemia"])
col_historico = obter_coluna(["familyhistorydiabetes", "family_history_diabetes", "historico_familiar"])
col_atividade = obter_coluna(["physicalactivitylevel", "physical_activity_level", "atividade_fisica"])
col_diabetes  = obter_coluna(["diabetes", "outcome", "target"])

df_silver = df_bronze \
    .withColumn("age", F.col("age").cast("int")) \
    .withColumn("bmi", F.col("bmi").cast("double")) \
    .withColumn("fasting_blood_sugar", F.col(col_glicose).cast("double") if col_glicose else F.lit(100.0)) \
    .withColumn(
        "family_history_diabetes",
        F.when(F.lower(F.trim(F.col(col_historico).cast("string"))).isin(["yes", "true", "1"]), 1)
         .when(F.lower(F.trim(F.col(col_historico).cast("string"))).isin(["no", "false", "0"]), 0)
         .otherwise(0) if col_historico else F.lit(0)
    ) \
    .withColumn(
        "diabetes",
        F.when(F.lower(F.trim(F.col(col_diabetes).cast("string"))).isin(["yes", "true", "1"]), 1)
         .when(F.lower(F.trim(F.col(col_diabetes).cast("string"))).isin(["no", "false", "0"]), 0)
         .otherwise(0) if col_diabetes else F.lit(0)
    ) \
    .withColumn("physical_activity_level", F.lower(F.trim(F.col(col_atividade).cast("string"))) if col_atividade else F.lit("moderate")) \
    .dropDuplicates() \
    .filter(F.col("age").isNotNull() & F.col("bmi").isNotNull()) \
    .drop("_ingestion_datetime", "_source_file")

df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABELA_SILVER)

In [0]:
%sql
SELECT
COUNT(*) AS total_registros,
SUM(CASE WHEN age IS NULL THEN 1 ELSE 0 END) AS nulos_idade,
SUM(CASE WHEN bmi IS NULL THEN 1 ELSE 0 END) AS nulos_imc
FROM mvp.staging.silver_diabetes_clean;

Camada Gold (Modelo Flat)

In [0]:
from pyspark.sql import functions as F

CATALOGO = "mvp"
ESQUEMA = "staging"

TABELA_SILVER = f"{CATALOGO}.{ESQUEMA}.silver_diabetes_clean"
TABELA_GOLD   = f"{CATALOGO}.{ESQUEMA}.gold_fato_diabetes"

df_silver = spark.table(TABELA_SILVER)

df_gold = df_silver \
    .withColumn("faixa_etaria", 
        F.when(F.col("age") < 30, "Jovem (<30)")
         .when((F.col("age") >= 30) & (F.col("age") <= 59), "Adulto (30-59)")
         .otherwise("Idoso (60+)")
    ) \
    .withColumn("categoria_bmi", 
        F.when(F.col("bmi") < 25.0, "1. Normal")
         .when((F.col("bmi") >= 25.0) & (F.col("bmi") < 30.0), "2. Sobrepeso")
         .otherwise("3. Obesidade")
    ) \
    .withColumn("categoria_glicose", 
        F.when(F.col("fasting_blood_sugar") < 100.0, "1. Normal")
         .when((F.col("fasting_blood_sugar") >= 100.0) & (F.col("fasting_blood_sugar") <= 125.0), "2. Alterada")
         .otherwise("3. Elevada")
    )

df_gold.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABELA_GOLD)

In [0]:
print(f"/Camada Gold criada com sucesso!Linhas: {spark.table(TABELA_GOLD).count()}")

Modelagem Dimensional Snowflake (SQL)

In [0]:
%sql
USE CATALOG mvp;
USE SCHEMA staging;

-- 1. Sub-dimensão IMC
CREATE OR REPLACE TABLE mvp.staging.dim_categoria_imc AS
SELECT 
    ROW_NUMBER() OVER (ORDER BY categoria_bmi) AS id_categoria_bmi,
    categoria_bmi AS descricao_categoria,
    CASE 
        WHEN categoria_bmi = '1. Normal' THEN 'Risco Baixo'
        WHEN categoria_bmi = '2. Sobrepeso' THEN 'Risco Moderado'
        ELSE 'Risco Elevado'
    END AS classificacao_risco_metabolico
FROM (SELECT DISTINCT categoria_bmi FROM mvp.staging.gold_fato_diabetes);

-- 2. Dimensão Pacientes
CREATE OR REPLACE TABLE mvp.staging.dim_paciente AS
SELECT 
    MONOTONICALLY_INCREASING_ID() AS id_paciente,
    g.age,
    g.faixa_etaria,
    g.family_history_diabetes,
    g.physical_activity_level,
    c.id_categoria_bmi
FROM mvp.staging.gold_fato_diabetes g
LEFT JOIN mvp.staging.dim_categoria_imc c ON g.categoria_bmi = c.descricao_categoria;

-- 3. Dimensão Glicemia
CREATE OR REPLACE TABLE mvp.staging.dim_glicemia AS
SELECT 
    ROW_NUMBER() OVER (ORDER BY categoria_glicose) AS id_glicemia,
    categoria_glicose AS faixa_glicemica
FROM (SELECT DISTINCT categoria_glicose FROM mvp.staging.gold_fato_diabetes);

-- 4. Tabela Fato Central
CREATE OR REPLACE TABLE mvp.staging.fato_diabetes_snowflake AS
SELECT 
    p.id_paciente,
    gl.id_glicemia,
    g.bmi,
    g.fasting_blood_sugar,
    g.diabetes AS flag_diabetes
FROM mvp.staging.gold_fato_diabetes g
INNER JOIN mvp.staging.dim_paciente p 
        ON g.age = p.age 
       AND g.family_history_diabetes = p.family_history_diabetes 
       AND g.physical_activity_level = p.physical_activity_level
INNER JOIN mvp.staging.dim_glicemia gl 
        ON g.categoria_glicose = gl.faixa_glicemica;

O retorno de "No rows returned" no Databricks/SQL é esperado e correto para os comandos de criação de tabelas, mas depende do tipo de comando executado.

Por que isso acontece:

Comandos DDL (CREATE OR REPLACE TABLE):
Ao executar instruções de definição de dados como CREATE OR REPLACE TABLE ... AS SELECT ..., o SGDB processa a instrução, cria a estrutura no catálogo e popula a tabela em segundo plano. O resultado padrão retornado pelo conector/interface SQL para esse tipo de comando é apenas um sinal de sucesso sem linhas exibidas (No rows returned ou OK).

Como validar se os dados foram inseridos:
Para confirmar que a modelagem gravou os registros corretamente nas tabelas da dimensão e fato, execute consultas de verificação (SELECT):

In [0]:
%sql
-- Verificar contagem de registros em cada tabela criada
SELECT 'dim_categoria_imc' AS tabela, COUNT(*) AS total FROM mvp.staging.dim_categoria_imc
UNION ALL
SELECT 'dim_paciente', COUNT(*) FROM mvp.staging.dim_paciente
UNION ALL
SELECT 'dim_glicemia', COUNT(*) FROM mvp.staging.dim_glicemia
UNION ALL
SELECT 'fato_diabetes_snowflake', COUNT(*) FROM mvp.staging.fato_diabetes_snowflake;

Se você rodar um SELECT simples nessas tabelas após a criação e o resultado continuar sendo "No rows returned", significa que o relacionamento dos JOINs falhou em encontrar correspondências.

Os dois pontos mais comuns para conferir caso o SELECT volte vazio:

Incompatibilidade nos JOINs da fato_diabetes_snowflake:
A tabela fato utiliza INNER JOIN cruzando colunas como age, family_history_diabetes, physical_activity_level e categoria_glicose. Se houver qualquer divergência de tipo de dado ou valores nulos (NULL) nessas colunas na camada Gold, o INNER JOIN filtrará 100% dos registros.

Dependência em cadeia:
Se a tabela mvp.staging.gold_fato_diabetes estiver vazia por conta de algum filtro na camada Silver/Bronze, todas as tabelas dimensionais Snowflake subsequentes também ficarão vazias.

In [0]:
# Script de Validação da Qualidade dos Dados (Executado no Databricks)
from pyspark.sql.functions import col, sum

df_raw = spark.table("mvp.staging.bronze_diabetes_raw")
df_clean = spark.table("mvp.staging.silver_diabetes_clean")

print(f"Total de registros Bronze (Raw): {df_raw.count()}")
print(f"Total de registros Silver (Clean): {df_clean.count()}")
print(f"Registros removidos (Duplicados/Nulos): {df_raw.count() - df_clean.count()}")

# Verificação de Nulos por coluna na camada Silver
df_clean.select([sum(col(c).isNull().cast("int")).alias(c) for c in df_clean.columns]).show()